In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import seaborn as sns
import random

## Events and Random Variables

Probability theory is the mathematical framework for quantifying uncertainty about what *might* happen. We will talk about things that do happen as **events**. If there is uncertainty about which of multiple possible events will occur, we will think about a data generating process that realizes one of those possible events as a **probability distribution**, or just distribution for short. We may think some events are more likely than others to occur, so we will say they have higher probability.

In other lessons, we will consider statistical inference. That uses observed data to look backwards and make inferences about the plausibility of alternative probability distributions as generators of the dataset that you've observed. But that's for another lesson. Here, we will just try to understand probability distributions, not make statistical inferences.

As you read about in the textbook, there are multiple ways of thinking about what a probability distribution represents. One, the frequentist interpretation, imagines running the data generating process a very many times and observing the frequencies of the different events. (More precisely, for those of you who studied real analysis, it's the limit as the number of times approaches infinity, but I'm going to be a little loose in this exposition and just talk about "very many times" as if it were infinity.) The frequentist approach matches very nicely with the code we can write in python. We will represent a probability distribution as a function, a data generator that we can call to generate a simulated outcome. We can call it a lot of times and look at the pattern of how frequently different values are returned.

The other common way of thinking about probability distributions is as a representation of beliefs. The data generation process may only ever occur one time. Think of the outcome of a sporting event. But we express our belief that there's an 80% chance that the Michigan Wolverines will win the football game. Frequentists have tried to stretch and imagine there's an infinite set of possible universes. In 80% of those, Michigan wins. We inhabit only one of those universes but we have uncertainty about which one. I think it's more natural to just think of the probability as a representation of beliefs. I *think* there's an 80% that Michigan will win. For this lesson, we won't talk much about probability as beliefs, but we will return to the idea when we consider Bayesian statistical inference, in another lesson.

## Coin Flips: The Bernoulli Distribution
Let's start by considering the so-called Bernoulli distribution. It's a great way to think about coin flips.

- There are only two possible events: the flip comes out heads, or it comes out tails.
- If we run this process a lot of times, we can observe the frequency of heads and tails.

In [ ]:
# flip a fair coin
random.seed(42)
random.random() > 0.5

In [ ]:
# flip a fair coin 20 times
[random.random() > 0.5 for _ in range(20)]

## Using the scipy library

The scipy library has built-in functions for making multiple random draws from many distributions, including Bernoulli.
The implementation of bernouuli has 0 and 1 as the two possible events, rather than True and False
With p, the probability of success (1) as the only parameter.

In [ ]:
from scipy.stats import bernoulli
np.random.seed(42)
# Generate the random draws
draws = bernoulli.rvs(p=0.5, size=20)

# Create a pandas DataFrame
df = pd.DataFrame(draws, columns=["Outcome"])

df

In [ ]:
# compute the frequency of each outcome
df["Outcome"].value_counts()

In [ ]:
# Let's draw a much larger sample
draws = bernoulli.rvs(p=0.5, size=100000)
df = pd.DataFrame(draws, columns=["Outcome"])
df["Outcome"].value_counts()

The `.value_counts` method has a `normalize` parameter that can be set to True to compute the relative frequencies (i.e., probabilities)

(Note that probabilities are always between 0 and 1, and they sum to 1.)

In [ ]:
df["Outcome"].value_counts(normalize=True)

In [ ]:
counts = df["Outcome"].value_counts()
freqs = df["Outcome"].value_counts(normalize=True)

# Create a figure and a single subplot
fig, ax1 = plt.subplots()

# Create the bar plot basd on counts
sns.barplot(x=counts.index, y=counts.values, ax=ax1)

# Also show the normalized frequency
# Create a second y-axis that shares the same x-axis
ax2 = ax1.twinx()
sns.barplot(x=freqs.index, y=freqs.values, ax=ax2)


# Set the maximum value of the y-axes
ax1.set_ylim(0, df.shape[0])  # max is the number of rows
ax2.set_ylim(0, 1)  # 1 is the maximum value a probability can have

# Show the plot
plt.show()

Let's make a pair of functions to do that work for us. We'll be able to call it repeatedly with slightly different inputs. 

In [ ]:
def draw_bernoulli(p, n):
    # Generate the random draws
    draws = bernoulli.rvs(p, size=n)
    # Create a pandas DataFrame
    return pd.DataFrame(draws, columns=["Outcome"])

def plot(data, ax1):
    # Get the counts and frequencies
    counts = data["Outcome"].value_counts()
    # Create the bar plot on the first y-axis
    ax1.set_ylim(0, data.shape[0])  # max is the number of rows
    counts = data["Outcome"].value_counts()
    sns.barplot(x=counts.index, y=counts, ax=ax1)
    ax1.set(xlabel=None, ylabel=None)  # Hide x-axis and y-axis legends

    # Plot the normalized frequency on a second y-axis, on the right side
    ax2 = ax1.twinx()
    ax2.set_ylim(0, 1)  # 1 is the maximum value a probability can have
    freqs = data["Outcome"].value_counts(normalize=True)
    sns.barplot(x=freqs.index, y=freqs.values, ax=ax2)
    ax2.set(xlabel=None, ylabel=None)  # Hide x-axis and y-axis legends


In [ ]:
df = draw_bernoulli(0.25, 100000)
df.head(20)

In [ ]:
df["Outcome"].value_counts()

In [ ]:
df["Outcome"].value_counts(normalize=True)

In [ ]:
plot(df, plt.gca())

Now let's generate plots for increasing values of p, and show them as small multiples in a larger plot

In [ ]:
# Create a larger figure
fig, axs = plt.subplots(3, 3, figsize=(15, 15))

# Generate the plots
for i, p in enumerate(np.linspace(0, 1, 9)):
    # first three on the first row; column 1 has the first plot, column 2 has the second plot, etc.
    plot(draw_bernoulli(p, 100000), axs[i // 3, i % 3])
    axs[i // 3, i % 3].set_title(f'p = {p:.2f}')  # Add a title indicating the value of p


# Adjust the layout
plt.tight_layout()
# Show the plots
plt.show()